STEP 1

In [0]:
%pip install mlflow

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd

In [0]:
path = "/Volumes/gr5069/raw/f1_data"

STEP 2

In [0]:
' /Volumes/gr5069/raw/airbnb/airbnb-cleaned-mlflow.csv'

In [0]:
df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True,
    inferSchema=True
)

display(df_results)

STEP 3

In [0]:
from pyspark.sql.functions import when

df_results = df_results.withColumn(
    "scored_points",
    when(df_results.points > 0, 1).otherwise(0)
)

display(df_results)

In [0]:
print("df_results:", df_results.columns)
print("df_model:", df_model.columns)

STEP 4

In [0]:
df_model = df_results.select(
    "grid",
    "driverId",
    "constructorId",
    "laps",
    "statusId",
    "scored_points"
).dropna()

display(df_model)

STEP 5

In [0]:
df = df_model.toPandas()

X = df.drop("scored_points", axis=1)
y = df["scored_points"]

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

STEP 6

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("accuracy:", accuracy_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("f1:", f1_score(y_test, y_pred))

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("accuracy:", accuracy_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("f1:", f1_score(y_test, y_pred))

STEP 7

In [0]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

mlflow.set_experiment("/Users/qt2167@columbia.edu/f1_mlflow_experiment")

with mlflow.start_run(run_name="rf_baseline"):
    n_estimators = 100
    max_depth = 5
    random_state = 42

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(rf, "random_forest_model")

    pred_df = pd.DataFrame({
        "actual": y_test.reset_index(drop=True),
        "predicted": y_pred
    })
    pred_df.to_csv("/tmp/predictions.csv", index=False)
    mlflow.log_artifact("/tmp/predictions.csv")

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.savefig("/tmp/confusion_matrix.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("/tmp/confusion_matrix.png")

    print("accuracy:", acc)
    print("precision:", prec)
    print("recall:", rec)
    print("f1_score:", f1)

This step logs a baseline Random Forest model using MLflow.

STEP 8

In [0]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

mlflow.set_experiment("/Users/qt2167@columbia.edu/f1_mlflow_experiment")

param_grid = [
    (50, 3),
    (50, 5),
    (50, 7),
    (100, 3),
    (100, 5),
    (100, 7),
    (150, 3),
    (150, 5),
    (150, 7),
    (200, 5)
]

for n_estimators, max_depth in param_grid:
    with mlflow.start_run(run_name=f"rf_n{n_estimators}_d{max_depth}"):
        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )

        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("random_state", 42)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)

        mlflow.sklearn.log_model(rf, name="random_forest_model")

        pred_df = pd.DataFrame({
            "actual": y_test.reset_index(drop=True),
            "predicted": y_pred
        })
        pred_file = f"/tmp/predictions_{n_estimators}_{max_depth}.csv"
        pred_df.to_csv(pred_file, index=False)
        mlflow.log_artifact(pred_file)

        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        plot_file = f"/tmp/confusion_matrix_{n_estimators}_{max_depth}.png"
        plt.savefig(plot_file, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_file)

        print(f"run finished: n_estimators={n_estimators}, max_depth={max_depth}, f1={f1:.4f}")

In this step, I performed hyperparameter tuning on the Random Forest model by testing multiple combinations of n_estimators and max_depth.

I ran experiments using MLflow to systematically compare model performance across different parameter settings. For each run, I logged evaluation metrics including accuracy, precision, recall, and F1-score, as well as model artifacts such as predictions and confusion matrices.

The goal of this step was to identify the combination of hyperparameters that yields the best predictive performance.

STEP 9

After running multiple experiments, I mainly focused on the F1 score to decide which model works best.

Among all runs, the combination of n_estimators = 50 and max_depth = 7 gave the highest F1 score.

Some models had similar accuracy, but their F1 scores were lower, which means they were not as balanced.

So I picked this one as the final model since it performs more consistently across different evaluation metrics.

STEP 10

In this project, I built a machine learning pipeline to predict whether a driver scores points in a race.

I started by preprocessing the F1 dataset and creating a binary target variable. Then, I trained a baseline Random Forest classifier and evaluated its performance.

Using MLflow, I tracked experiments and compared different hyperparameter configurations. After tuning, I identified the best-performing model based on F1-score.

Overall, the Random Forest model performed well, achieving strong accuracy and balanced precision-recall performance. Future improvements could include feature engineering or trying different model types such as gradient boosting.

In [0]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

display(importance)

The most important features are grid position and laps, indicating that starting position plays a key role in scoring points.